# Week 3: training-set scaling
Author: Selina.

How does AP change with 10–100 training recordings? Change from Week 2: use deterministic nested, approximately family-stratified training subsets; keep validation and test fixed. Use CNN/8, BCE, Adam 0.0003, seed 163, train-only normalization and the same stopping rule. Each size selects a validation-AP checkpoint and validation-F1 threshold. This is a descriptive scaling curve; do not pick a training size using test AP.


In [ ]:
from cs163_rfi import paths, scaling
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = scaling.OUT
SIZES = scaling.SIZES
RUN_TRAINING_AND_TEST = False


In [ ]:
if RUN_TRAINING_AND_TEST:
    subsets = scaling.prepare()
    display(pd.DataFrame({"train_recordings": SIZES, "families_present": [subsets[n].rfi_type.nunique() for n in SIZES]}))
else:
    OUT.mkdir(parents=True, exist_ok=True)


## Train the prespecified sizes
This reproduces the historical protocol, which evaluates test for each frozen size. Enable only after fixing the full experiment design.


In [ ]:
if RUN_TRAINING_AND_TEST:
    summary = pd.DataFrame([scaling.train_size(n) for n in SIZES])
    summary.to_csv(OUT / "scaling_summary.csv", index=False)
else:
    summary = pd.read_csv(paths.RESULTS_DIR / "reference/scaling_summary.csv")
display(summary)


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(summary.train_recordings,summary.validation_ap,
         marker="o",label="Validation AP")
plt.plot(summary.train_recordings,summary.test_ap,
         marker="o",label="Test AP")
plt.xlabel("Training recordings")
plt.ylabel("Pixel AP")
plt.title("Performance vs training-set size")
plt.legend(); plt.grid(alpha=.2); plt.tight_layout()
plt.savefig(OUT/"scaling_ap.png",dpi=200)
plt.show()

plt.figure(figsize=(7,4))
plt.plot(summary.train_recordings,summary.test_f1,marker="o")
plt.xlabel("Training recordings")
plt.ylabel("Test F1 (validation-selected threshold)")
plt.title("F1 vs training-set size")
plt.grid(alpha=.2); plt.tight_layout()
plt.savefig(OUT/"scaling_f1.png",dpi=200)
plt.show()

print("Saved:",OUT)

Small subsets are noisy; increases are not monotonic. Gains diminish around 70–100 recordings in this single-seed experiment.


In [ ]:
print("Saved:", OUT)
